# Real Estate Data Exploration


This notebook explains the **1,689-row model dataset** with simple tables and one
chart per cell. It checks missing values and outliers, compares property and
location groups, and reviews which numeric features move most closely with price.

The data describes current online asking prices. It does not show completed sale
prices or future market movements.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

TARGET = "price_usd"
CATEGORY_COLORS = ["#2563EB", "#F59E0B", "#6B7D2A", "#D69E2E", "#D97777"]
CORRELATION_CMAP = sns.diverging_palette(30, 250, s=75, l=65, as_cmap=True)
NON_ANALYSIS_NAMES = {
    "source_site", "source_name", "scraped_at", "scrape_date", "crawl_date",
    "title", "description", "raw_text", "currency", "original_price",
    "listing_id", "property_id", "external_id",
    "price_per_sqm", "price_usd_per_sqm",
}

def is_non_analysis_column(column):
    """Identify metadata or leakage fields excluded from EDA calculations."""
    return column in NON_ANALYSIS_NAMES or column.endswith("_url") or column.startswith("source_")

## 1. Load the Dataset

Load the compact model-ready CSV. The path works from the project root or the
`code` folder.

In [2]:
data_candidates = [
    Path("..") / "data" / "scraped_real_estate_model_features.csv",
    Path("data") / "scraped_real_estate_model_features.csv",
]
DATA_PATH = next((path for path in data_candidates if path.exists()), data_candidates[0])

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns:
    raise ValueError(f"Required target column is missing: {TARGET}")

print(f"Source: {DATA_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")

Source: ..\data\scraped_real_estate_model_features.csv
Rows: 1,689
Columns: 63


In [3]:
df.head()

,price_usd,property_type,property_subtype,publication_type,construction_stage,location_key,country,province,city,neighborhood,latitude,longitude,orientation,effective_area_sqm,covered_area_sqm,total_area_sqm,semicovered_area_sqm,uncovered_area_sqm,area_per_room_sqm,area_bucket,bedrooms,bathrooms,total_rooms,room_bucket,toilets,bathroom_ratio,parking_spaces,has_parking,floor_number,floors_in_building,is_high_floor,age_years,age_bucket,floor_bucket,expenses_usd,distance_to_sea_blocks,amenity_count,is_apartment,is_house,is_new_construction,is_under_construction,has_balcony,has_terrace,has_garden,has_patio,has_pool,has_elevator,has_security,has_storage_room,has_laundry_room,has_air_conditioning,has_heating,has_grill,has_gym,has_doorman,is_furnished,is_gated_community,is_near_beach,is_near_park,is_near_sea,is_near_subway,is_owner_direct,pets_allowed
0,260000,House,NaN,NaN,NaN,AR | Villa Devoto | Capital Federal | Venta Vi...,AR,Villa Devoto,Capital Federal,Venta Villa Devoto,NaN,NaN,NaN,249.0,249.0,NaN,NaN,NaN,62.25,06_luxury_scale,3.0,2.0,4.0,04_three_bedroom,NaN,0.67,NaN,0,4.0,NaN,0,NaN,NaN,03_mid,NaN,NaN,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,450000,House,NaN,NaN,NaN,AR | San Cristobal | Capital Federal,AR,San Cristobal,Capital Federal,NaN,NaN,NaN,NaN,297.0,297.0,NaN,NaN,NaN,59.40,06_luxury_scale,4.0,3.0,5.0,05_four_plus_bedroom,1.0,0.75,1.0,1,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,269000,House,NaN,NaN,NaN,AR | Adrogué | Almirante Brown,AR,Adrogué,Almirante Brown,NaN,NaN,NaN,NaN,357.0,357.0,NaN,NaN,NaN,89.25,06_luxury_scale,3.0,2.0,4.0,04_three_bedroom,1.0,0.67,2.0,1,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,3,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,158000,House,NaN,NaN,new,AR | Los Pinos | Exaltación de la Cruz,AR,Los Pinos,Exaltación de la Cruz,NaN,NaN,NaN,NaN,181.0,181.0,NaN,NaN,NaN,45.25,05_extra_large,3.0,3.0,4.0,04_three_bedroom,NaN,1.00,NaN,0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,299000,House,NaN,NaN,NaN,AR | Campos de Echeverría | Canning | Campos d...,AR,Campos de Echeverría,Canning,Campos de Echeverría. Canning,NaN,NaN,NaN,200.0,200.0,NaN,NaN,NaN,40.00,05_extra_large,4.0,2.0,5.0,05_four_plus_bedroom,NaN,0.50,1.0,1,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,5,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 2. Data Quality

Keep the original dataframe unchanged. Use a copy without exact duplicates so a
repeated row does not receive extra weight in the analysis.

In [4]:
non_analysis_columns = [column for column in df.columns if is_non_analysis_column(column)]
analysis_columns = [column for column in df.columns if column not in non_analysis_columns]
duplicate_copies = int(df.duplicated(subset=analysis_columns).sum())
rows_in_duplicate_groups = int(df.duplicated(subset=analysis_columns, keep=False).sum())
all_missing_columns = df[analysis_columns].columns[df[analysis_columns].isna().all()].tolist()
analysis_df = df.drop_duplicates(subset=analysis_columns)[analysis_columns].copy()

quality_summary = pd.Series({
    "raw_rows": len(df),
    "rows_after_feature_target_deduplication": len(analysis_df),
    "duplicate_copies": duplicate_copies,
    "rows_in_duplicate_groups": rows_in_duplicate_groups,
    "analysis_columns": len(analysis_columns),
    "excluded_metadata_or_leakage_columns": len(non_analysis_columns),
    "numeric_columns": analysis_df.select_dtypes(include="number").shape[1],
    "categorical_columns": analysis_df.select_dtypes(exclude="number").shape[1],
    "all_missing_columns": len(all_missing_columns),
    "missing_target_values": int(df[TARGET].isna().sum()),
}, name="value")

quality_summary.to_frame()

,value
raw_rows,1689
rows_after_feature_target_deduplication,1689
duplicate_copies,0
rows_in_duplicate_groups,0
analysis_columns,63
excluded_metadata_or_leakage_columns,0
numeric_columns,49
categorical_columns,14
all_missing_columns,2
missing_target_values,0


In [5]:
missing_table = (
    df.isna().sum().to_frame("missing_count")
    .assign(missing_percent=lambda table: table["missing_count"] / len(df) * 100)
    .sort_values(["missing_percent", "missing_count"], ascending=False)
)

missing_table.head(20).round(1)

,missing_count,missing_percent
floors_in_building,1689,100.0
distance_to_sea_blocks,1689,100.0
semicovered_area_sqm,1680,99.5
uncovered_area_sqm,1673,99.1
orientation,1596,94.5
age_years,1562,92.5
age_bucket,1562,92.5
publication_type,1508,89.3
property_subtype,1506,89.2
toilets,1471,87.1


In [6]:
missing_plot = missing_table[missing_table["missing_count"] > 0].head(15).sort_values("missing_percent")

if missing_plot.empty:
    print("No missing values found.")
else:
    plt.figure(figsize=(9, 6))
    plt.barh(missing_plot.index, missing_plot["missing_percent"], color="#F59E0B")
    plt.title("Features with the Most Missing Data")
    plt.xlabel("Missing values (%)")
    plt.ylabel("Feature")
    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\2348899636.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Price Distribution and Outliers

The first chart uses a log scale so the full price range is visible. The second
shows the main range below the 99th percentile. Expensive listings are reviewed,
not silently removed.

In [7]:
price_summary = analysis_df[TARGET].describe(percentiles=[0.50, 0.90, 0.95, 0.99])
price_outlier_threshold = analysis_df[TARGET].quantile(0.99)
high_price_outliers = analysis_df[analysis_df[TARGET] > price_outlier_threshold].copy()
correlation_df = analysis_df[analysis_df[TARGET] <= price_outlier_threshold].copy()

print(f"99th-percentile review threshold: {price_outlier_threshold:,.0f} USD")
print(f"Rows above the threshold: {len(high_price_outliers):,}")
price_summary.round(2)

99th-percentile review threshold: 2,076,800 USD
Rows above the threshold: 17


count        1689.00
mean       328606.52
std        678208.70
min         40000.00
50%        194000.00
90%        600000.00
95%        880000.00
99%       2076800.00
max      21060000.00
Name: price_usd, dtype: float64

In [8]:
plt.figure(figsize=(9, 5))
sns.histplot(analysis_df[TARGET].dropna(), bins=45, color="#2563EB")
plt.xscale("log")
plt.title("Full Price Distribution (Log Scale)")
plt.xlabel("Price (USD, log scale)")
plt.ylabel("Properties")
plt.tight_layout()
plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\1341370435.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
plt.figure(figsize=(9, 5))
sns.histplot(correlation_df[TARGET].dropna(), bins=45, color="#2563EB")
plt.title("Price Distribution at or Below the 99th Percentile")
plt.xlabel("Price (USD)")
plt.ylabel("Properties")
plt.tight_layout()
plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\3948964305.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
preferred_outlier_columns = [
    TARGET, "property_type", "country", "city",
    "effective_area_sqm", "covered_area_sqm", "bedrooms", "bathrooms",
]
outlier_review_columns = [column for column in preferred_outlier_columns if column in df.columns]

high_price_outliers[outlier_review_columns].sort_values(TARGET, ascending=False).head(20)

,price_usd,property_type,country,city,effective_area_sqm,covered_area_sqm,bedrooms,bathrooms
857,21060000,Apartment,UY,Punta del Este,NaN,NaN,6.0,8.0
856,10530000,Apartment,UY,Punta del Este,810.0,810.0,5.0,6.0
855,5460000,Apartment,UY,Punta del Este,420.0,420.0,3.0,4.0
64,5000000,House,AR,San Fernando,NaN,NaN,4.0,NaN
46,3900000,House,AR,Palermo,312.0,312.0,5.0,5.0
1248,3400000,Apartment,AR,"Capital Federal, Argentina,",310.0,310.0,NaN,4.0
212,2800000,Apartment,UY,NaN,250.0,250.0,2.0,2.0
55,2800000,House,AR,Nordelta,1060.0,1060.0,4.0,5.0
41,2700000,House,AR,Capital Federal,550.0,550.0,4.0,4.0
1189,2600000,House,UY,NaN,800.0,800.0,1.0,3.0


## 4. Property and Location Mix

Show the main property and location groups. The code chooses available columns
automatically so small schema changes do not break the notebook.

In [11]:
categorical_columns = analysis_df.select_dtypes(exclude="number").columns.tolist()
main_category = "property_type" if "property_type" in analysis_df.columns else (categorical_columns[0] if categorical_columns else None)
main_category

'property_type'

In [12]:
if main_category is None:
    print("No categorical feature is available.")
else:
    category_order = analysis_df[main_category].fillna("Unknown").value_counts().head(12).index
    category_counts = analysis_df[main_category].fillna("Unknown").value_counts().loc[category_order]
    plt.figure(figsize=(9, 5))
    plt.bar(category_counts.index.astype(str), category_counts.values, color="#2563EB")
    plt.title(f"Rows by {main_category.replace('_', ' ').title()}")
    plt.xlabel(main_category.replace("_", " ").title())
    plt.ylabel("Properties")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\3063686161.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
if main_category is None:
    print("No categorical feature is available.")
else:
    category_order = correlation_df[main_category].fillna("Unknown").value_counts().head(10).index
    category_price_data = correlation_df.copy()
    category_price_data[main_category] = category_price_data[main_category].fillna("Unknown")
    category_price_data = category_price_data[category_price_data[main_category].isin(category_order)]
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=category_price_data, x=main_category, y=TARGET, order=category_order, color="#93C5FD")
    plt.title(f"Price by {main_category.replace('_', ' ').title()} (Below 99th Percentile)")
    plt.xlabel(main_category.replace("_", " ").title())
    plt.ylabel("Price (USD)")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\1840247307.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
location_candidates = ["city", "neighborhood", "province", "country"]
location_feature = next((column for column in location_candidates if column in analysis_df.columns), None)
location_feature

'city'

In [15]:
if location_feature is None:
    print("No location feature is available.")
else:
    location_counts = analysis_df[location_feature].dropna().value_counts().head(12).sort_values()
    plt.figure(figsize=(9, 6))
    plt.barh(location_counts.index.astype(str), location_counts.values, color="#6B7D2A")
    plt.title(f"Most Common {location_feature.replace('_', ' ').title()} Values")
    plt.xlabel("Properties")
    plt.ylabel(location_feature.replace("_", " ").title())
    plt.tight_layout()
    plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\1880717350.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
if location_feature is None:
    print("No location feature is available.")
else:
    common_locations = correlation_df[location_feature].dropna().value_counts().head(12).index
    median_price_by_location = (
        correlation_df[correlation_df[location_feature].isin(common_locations)]
        .groupby(location_feature)[TARGET]
        .median()
        .sort_values()
    )
    plt.figure(figsize=(9, 6))
    plt.barh(median_price_by_location.index.astype(str), median_price_by_location.values, color="#D69E2E")
    plt.title(f"Median Price in the Most Common {location_feature.replace('_', ' ').title()} Values")
    plt.xlabel("Median price (USD, below 99th percentile)")
    plt.ylabel(location_feature.replace("_", " ").title())
    plt.tight_layout()
    plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\2361218734.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Price Correlations

Pearson measures linear relationships. Spearman measures ranked relationships and
is less sensitive to extreme values. The table also checks the number of observed
pairs. Correlation describes association; it does not prove cause.

In [17]:
numeric_columns = analysis_df.select_dtypes(include="number").columns.tolist()
minimum_pairs = max(50, int(len(correlation_df) * 0.25))

pearson_all = analysis_df[numeric_columns].corr(method="pearson")[TARGET].drop(TARGET)
spearman_all = analysis_df[numeric_columns].corr(method="spearman")[TARGET].drop(TARGET)
pearson_below_p99 = correlation_df[numeric_columns].corr(method="pearson")[TARGET].drop(TARGET)
observed_pairs = correlation_df[numeric_columns].notna().sum().drop(TARGET)

price_correlations = pd.DataFrame({
    "pearson_all_prices": pearson_all,
    "pearson_below_p99": pearson_below_p99,
    "spearman_all_prices": spearman_all,
    "observed_pairs_below_p99": observed_pairs,
})
price_correlations = price_correlations[
    (price_correlations["observed_pairs_below_p99"] >= minimum_pairs)
    & price_correlations["pearson_below_p99"].notna()
]
reliable_price_correlations = price_correlations.sort_values(
    "pearson_below_p99", key=lambda values: values.abs(), ascending=False
)

In [18]:
print(f"Minimum observed pairs required: {minimum_pairs:,}")
price_correlations.head(20).round(3)

Minimum observed pairs required: 418


,pearson_all_prices,pearson_below_p99,spearman_all_prices,observed_pairs_below_p99
latitude,-0.059,-0.070,-0.209,984
longitude,0.226,0.267,0.433,984
effective_area_sqm,0.540,0.587,0.749,1660
covered_area_sqm,0.541,0.588,0.751,1657
total_area_sqm,0.442,0.429,0.618,563
area_per_room_sqm,0.427,0.435,0.490,1651
bedrooms,0.338,0.572,0.698,1657
bathrooms,0.502,0.702,0.705,1666
total_rooms,0.519,0.516,0.687,610
bathroom_ratio,0.132,0.087,-0.072,1666


In [19]:
correlation_plot = reliable_price_correlations.head(12).sort_values("pearson_below_p99")
bar_colors = ["#F59E0B" if value < 0 else "#2563EB" for value in correlation_plot["pearson_below_p99"]]

plt.figure(figsize=(10, 6))
bars = plt.barh(correlation_plot.index, correlation_plot["pearson_below_p99"], color=bar_colors)
plt.axvline(0, color="#374151", linewidth=1)
plt.xlim(-1, 1)
plt.title("Strongest Reliable Numeric Correlations with Price")
plt.xlabel("Pearson correlation (prices at or below 99th percentile)")
plt.ylabel("Feature")
for bar, value in zip(bars, correlation_plot["pearson_below_p99"]):
    label_x = value + 0.02 if value >= 0 else value - 0.02
    alignment = "left" if value >= 0 else "right"
    plt.text(label_x, bar.get_y() + bar.get_height() / 2, f"{value:.2f}", va="center", ha=alignment)
plt.tight_layout()
plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\3894968787.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
heatmap_features = reliable_price_correlations.head(9).index.tolist() + [TARGET]

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_df[heatmap_features].corr(),
    annot=True,
    fmt=".2f",
    cmap=CORRELATION_CMAP,
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
)
plt.title("Correlation Matrix for the Strongest Price Features")
plt.tight_layout()
plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\2061634565.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Strongest Feature Relationships

Choose the four strongest reliable numeric relationships automatically. Small
integer features use box plots; continuous features use scatter plots.

In [21]:
def plot_feature_vs_price(feature):
    """Plot one reliable feature against price in a separate figure.

    Continuous features use scatter plots and low-cardinality numeric features
    use box plots so each relationship stays readable.
    """
    plot_columns = [feature, TARGET]
    if main_category is not None:
        plot_columns.append(main_category)
    plot_data = correlation_df[plot_columns].dropna(subset=[feature, TARGET]).copy()
    feature_label = feature.replace("_", " ").title()

    plt.figure(figsize=(9, 5))
    if plot_data[feature].nunique() <= 8:
        sns.boxplot(data=plot_data, x=feature, y=TARGET, color="#93C5FD")
    else:
        hue_column = main_category if main_category is not None and plot_data[main_category].nunique() <= 10 else None
        hue_palette = None
        if hue_column is not None:
            hue_values = sorted(plot_data[hue_column].dropna().unique())
            hue_palette = {value: CATEGORY_COLORS[index % len(CATEGORY_COLORS)] for index, value in enumerate(hue_values)}
        sns.scatterplot(
            data=plot_data, x=feature, y=TARGET, hue=hue_column,
            alpha=0.55, edgecolor="none", palette=hue_palette,
        )
        if hue_column is not None:
            plt.legend(title=hue_column.replace("_", " ").title(), bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.title(f"Price vs {feature_label} (n={len(plot_data):,})")
    plt.xlabel(feature_label)
    plt.ylabel("Price (USD, below 99th percentile)")
    plt.tight_layout()
    plt.show()

In [22]:
top_price_features = reliable_price_correlations.head(4).index.tolist()
top_price_features

['bathrooms', 'covered_area_sqm', 'effective_area_sqm', 'bedrooms']

In [23]:
plot_feature_vs_price(top_price_features[0])

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\306009758.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
plot_feature_vs_price(top_price_features[1])

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\306009758.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
plot_feature_vs_price(top_price_features[2])

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\306009758.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
plot_feature_vs_price(top_price_features[3])

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\306009758.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Economic Context

The proposal also asks for economic indicators. The CSV below contains annual
World Bank inflation and GDP-growth values for Argentina and Uruguay. These values
give country-level context only; they are not matched to individual listings and
are not used by the price model.

In [27]:
macro_candidates = [
    Path("..") / "data" / "macroeconomic_indicators.csv",
    Path("data") / "macroeconomic_indicators.csv",
]
MACRO_PATH = next((path for path in macro_candidates if path.exists()), macro_candidates[0])
macro_df = pd.read_csv(MACRO_PATH)

macro_latest = (
    macro_df.sort_values("year")
    .groupby(["indicator", "country"], as_index=False)
    .tail(1)[["country", "indicator", "year", "value", "unit"]]
    .sort_values(["indicator", "country"])
)
print(f"Source: {MACRO_PATH}")
macro_latest

Source: ..\data\macroeconomic_indicators.csv


,country,indicator,year,value,unit
26,Argentina,GDP growth,2024,-1.3429,annual %
36,Uruguay,GDP growth,2024,3.3258,annual %
6,Argentina,"Inflation, consumer prices",2024,219.8839,annual %
16,Uruguay,"Inflation, consumer prices",2024,4.8491,annual %


In [28]:
inflation_data = macro_df[macro_df["indicator_code"] == "FP.CPI.TOTL.ZG"]

plt.figure(figsize=(9, 5))
for country, group in inflation_data.groupby("country"):
    group = group.sort_values("year")
    plt.plot(group["year"], group["value"], marker="o", label=country)
plt.title("Consumer Price Inflation")
plt.xlabel("Year")
plt.ylabel("Annual change (%)")
plt.legend(title="Country")
plt.tight_layout()
plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\4000210495.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [29]:
gdp_data = macro_df[macro_df["indicator_code"] == "NY.GDP.MKTP.KD.ZG"]

plt.figure(figsize=(9, 5))
for country, group in gdp_data.groupby("country"):
    group = group.sort_values("year")
    plt.plot(group["year"], group["value"], marker="o", label=country)
plt.axhline(0, color="#374151", linewidth=1)
plt.title("Real GDP Growth")
plt.xlabel("Year")
plt.ylabel("Annual change (%)")
plt.legend(title="Country")
plt.tight_layout()
plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_34204\3141308325.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Takeaways

- Property size, rooms, bathrooms, location, and expenses carry useful price signal.
- Review correlations together with observed pair counts and the high-price table.
- A binary `0` means “not mentioned,” not confirmed absence.
- The sample is dominated by Uruguay, InfoCasas, and apartments.
- Economic indicators provide context only. Future forecasting needs dated listing
  snapshots collected over time.